# PhysAR-IDS -- NB11b (v6): conformal prediction that sees the recent data (blocked cross-conformal)

**Why this notebook exists.** NB11 calibrated on the last 15 % of every (class, PLC) series and fitted the model on the earlier 85 %. That model never saw the most recent benign behaviour: benign recall fell from 0.978 (full-train model) to 0.776, benign coverage stuck at 0.78 in every seed, and the sets could not repair it because the model was confidently wrong about about 22 % of benign test rows. Conformal prediction inherits that shift.

**Fix tested here: blocked cross-conformal (CV+, Barber et al. 2021).** Every series is cut into K = 6 contiguous time blocks. For each block, a model is trained on all *other* blocks (with a 2 % gap either side of the held-out block, so the model sees the recent data whenever the held-out block is not the last one) and scores the held-out block. Calibration then uses the out-of-fold scores of **every** training row, and the test set is scored by all K models (CV+ rule: label y is in the set if fewer than (1 - alpha)(n + 1) out-of-fold scores are below the test score under the corresponding fold model). A class-conditional variant (`cvplus_cc`) uses only calibration rows of the label being tested.

**Methods compared (alpha 0.10, 0.05, 0.01):**
* `split_last` -- NB11's recipe reproduced inside this notebook: the model for the last block, threshold from the last block's out-of-fold scores (the baseline).
* `blocked | cvplus`, `blocked | cvplus_cc` -- the fix.
* `random | cvplus` -- control: same CV+ with folds assigned to random bursts. Calibration is then exchangeable *within* training, so any remaining shortfall on the test split is pure temporal shift that no calibration trick over the training data can remove.
* `full | pooledq` and `full | lastq` -- heuristic, no guarantee: the full-train model (NB2, the one with F1 0.8635) with a LAC threshold taken from the pooled blocked out-of-fold scores, or from the last block only. The full model saw the calibration rows, so these are practical approximations, not valid conformal sets.

**Also reported:** where the benign problem lives (out-of-fold recall per class and time block), point performance of the K-model ensemble vs the full model, and the same burst-cluster bootstrap CIs as NB11. **Not tested here:** behaviour under attack (NB11 section 7 applies to the split-conformal model only). **Limit to carry into the text:** CV+ guarantees 1 - 2 alpha under exchangeability, and the temporal test split is not exchangeable with the training blocks; read the result as an empirical coverage under shift.

**Needs:** `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`. **Runtime:** about 25-40 min on the local GPU (6 + 6 + 1 fits per seed).

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings, math
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
ALPHAS = [0.1, 0.05, 0.01]
GAP_FRAC = 0.02
N_BOOT = 2000
K_FOLDS = 6

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"}')
print('Working directory:', os.getcwd())

## 1. Load v6 data, bursts and time blocks

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(np.int64); y_test = np.asarray(data['y_test']).astype(np.int64)
plc_tr, plc_te = np.asarray(data['PLCID_train']), np.asarray(data['PLCID_test']); ts_tr, ts_te = np.asarray(data['timestamp_train']), np.asarray(data['timestamp_test'])
N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
for k_ in ('n_estimators', 'max_depth', 'min_child_weight'):
    if k_ in best_params: best_params[k_] = int(best_params[k_])
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

def burst_ids(y, plc, ts): return pd.DataFrame({'y': y, 'p': plc, 't': ts}).groupby(['y', 'p', 't'], sort=False).ngroup().to_numpy()
bid_tr, bid_te = burst_ids(y_train, plc_tr, ts_tr), burst_ids(y_test, plc_te, ts_te)
_df = pd.DataFrame({'y': y_train, 'p': plc_tr, 't': ts_tr}); _g = _df.groupby(['y', 'p'])['t']
frac = ((_g.rank(method='dense') - 1) / _g.transform('nunique')).to_numpy()
block = np.minimum((frac * K_FOLDS).astype(int), K_FOLDS - 1)
print(f'train {len(y_train):,} | test {len(y_test):,} | blocks {K_FOLDS}: rows per block {np.bincount(block).tolist()} | test bursts {len(np.unique(bid_te))}')
assert all(len(set(np.unique(bid_tr[block == k])) & set(np.unique(bid_tr[block != k]))) == 0 for k in range(K_FOLDS)), 'a burst straddles two blocks'

## 2. Helpers

In [ ]:
def fit_pair(X, y, seed):
    # GPU-fitted classifier (for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_nb11_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu
print('helpers defined')

In [ ]:
def qhat(s, alpha):
    n = len(s); k = int(math.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else float(np.sort(s)[k - 1])
def eval_sets(sets, y, bid, boot_seed=0):
    n = len(y); covered = sets[np.arange(n), y]; size = sets.sum(1); out = {}
    out['cov'] = float(covered.mean()); out['size'] = float(size.mean()); out['single'] = float((size == 1).mean()); out['empty'] = float((size == 0).mean())
    out['cov_pc'] = np.array([covered[y == c].mean() if (y == c).any() else np.nan for c in range(6)]); out['size_pc'] = np.array([size[y == c].mean() if (y == c).any() else np.nan for c in range(6)])
    out['worst_cov'] = float(np.nanmin(out['cov_pc'][np.bincount(y, minlength=6) > 0]))
    out['only_benign'] = float(((size == 1) & sets[:, 0]).mean()); out['has_benign'] = float(sets[:, 0].mean())
    u, inv = np.unique(bid, return_inverse=True); nb = len(u); cb = np.bincount(inv, covered.astype(float), nb); nr = np.bincount(inv, minlength=nb).astype(float)
    idx = np.random.RandomState(boot_seed).randint(nb, size=(N_BOOT, nb)); cs = cb[idx].sum(1) / nr[idx].sum(1)
    out['cov_lo'], out['cov_hi'] = [float(v) for v in np.percentile(cs, [2.5, 97.5])]; out['n_bursts'] = nb
    p = out['cov']; out['cov_row_hw'] = float(1.96 * math.sqrt(max(p * (1 - p), 1e-12) / n))
    return out
def conf_by_burst(y, pred, bid):
    u, inv = np.unique(bid, return_inverse=True); return np.bincount(inv * 36 + y * 6 + pred, minlength=len(u) * 36).reshape(len(u), 6, 6).astype(float)
def f1_from_conf(C):
    tp = np.einsum('...ii->...i', C); rs = C.sum(-1); cs = C.sum(-2)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rs + cs > 0, 2 * tp / (rs + cs), 0.0); rec = np.where(rs > 0, tp / rs, np.nan)
    sup = rs > 0; macro = (f1 * sup).sum(-1) / sup.sum(-1); weighted = (f1 * rs).sum(-1) / rs.sum(-1)
    return macro, weighted, f1, rec
print('helpers defined')

## 3. Cross-conformal machinery

`cvplus_sets` applies the CV+ rule to the test scores of the K fold models; `cvplus_cc_sets` is the class-conditional version.

In [ ]:
def prep_oof(R_oof, y_tr, fold):
    # sorted out-of-fold scores per fold (all classes) and per fold and class
    A = [np.sort(R_oof[fold == k]) for k in range(K_FOLDS)]
    B = [[np.sort(R_oof[(fold == k) & (y_tr == c)]) for c in range(6)] for k in range(K_FOLDS)]
    return A, B
def cvplus_sets(P_list, A, n, alpha):
    cnt = np.zeros(P_list[0].shape, np.int64)
    for k, P in enumerate(P_list): cnt += np.searchsorted(A[k], (1.0 - P).ravel(), side='left').reshape(P.shape)
    return cnt < (1 - alpha) * (n + 1)
def cvplus_cc_sets(P_list, B, n_c, alpha):
    cnt = np.zeros(P_list[0].shape, np.int64)
    for k, P in enumerate(P_list):
        for c in range(6): cnt[:, c] += np.searchsorted(B[k][c], 1.0 - P[:, c], side='left')
    return cnt < ((1 - alpha) * (np.asarray(n_c) + 1))[None, :]
print('cross-conformal machinery defined')

## 4. Per-seed run

In [ ]:
METHOD_ROWS = []   # (scheme, method) labels in the order they are stored
RES = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'conformalfix_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, seed, ALPHAS, K_FOLDS, GAP_FRAC, N_BOOT, float(X_train[:5].sum()), len(y_train), SMOKE_TEST, 'nb11b-v1'], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {}; rng = np.random.RandomState(seed + 2200)
    full_gpu, full = fit_pair(X_train, y_train, seed); del full_gpu; Pfull = full.predict_proba(X_test)
    mac, wei, f1c, rec = f1_from_conf(conf_by_burst(y_test, Pfull.argmax(1), bid_te).sum(0)); r['perf|full|macro'] = float(mac); r['perf|full|weighted'] = float(wei); r['perf|full|recall_pc'] = rec
    ub = np.unique(bid_tr); fold_rand_of = dict(zip(ub, rng.permutation(len(ub)) % K_FOLDS)); fold_rand = np.array([fold_rand_of[b] for b in bid_tr])
    schemes = {'blocked': block, 'random': fold_rand}; OOF = {}; PT = {}
    for sch, fold in schemes.items():
        P_oof = np.zeros((len(y_train), 6), np.float32); P_list = []
        for k in range(K_FOLDS):
            if sch == 'blocked': tr = ~((frac >= k / K_FOLDS - GAP_FRAC) & (frac < (k + 1) / K_FOLDS + GAP_FRAC))
            else: tr = fold != k
            g_, m_ = fit_pair(X_train[tr], y_train[tr], seed * 100 + k); del g_
            ho = fold == k; P_oof[ho] = m_.predict_proba(X_train[ho]); P_list.append(m_.predict_proba(X_test))
        OOF[sch], PT[sch] = P_oof, P_list
        R = 1.0 - P_oof[np.arange(len(y_train)), y_train]; A, B = prep_oof(R, y_train, fold); n = len(R); n_c = np.bincount(y_train, minlength=6)
        r[f'oofrecall|{sch}'] = np.array([[(P_oof[(fold == k) & (y_train == c)].argmax(1) == c).mean() for c in range(6)] for k in range(K_FOLDS)])
        Pens = np.mean(P_list, 0); mac, wei, f1c, rec = f1_from_conf(conf_by_burst(y_test, Pens.argmax(1), bid_te).sum(0)); r[f'perf|ens_{sch}|macro'] = float(mac); r[f'perf|ens_{sch}|weighted'] = float(wei); r[f'perf|ens_{sch}|recall_pc'] = rec
        for a in ALPHAS:
            for meth, sets in [('cvplus', cvplus_sets(P_list, A, n, a)), ('cvplus_cc', cvplus_cc_sets(P_list, B, n_c, a))]:
                for k_, v in eval_sets(sets, y_test, bid_te).items(): r[f'{sch}|{meth}|{a:g}|{k_}'] = v
            if sch == 'blocked':
                Rl = R[block == K_FOLDS - 1]; Pl = P_list[K_FOLDS - 1]
                ql = qhat(Rl, a); qp = qhat(R, a)
                for meth, sets in [('split_last', (1 - Pl) <= ql), ('full_lastq', (1 - Pfull) <= ql), ('full_pooledq', (1 - Pfull) <= qp)]:
                    for k_, v in eval_sets(sets, y_test, bid_te).items(): r[f'blocked|{meth}|{a:g}|{k_}'] = v
                r[f'blocked|q|{a:g}'] = np.array([ql, qp])
    mac, wei, f1c, rec = f1_from_conf(conf_by_burst(y_test, PT['blocked'][K_FOLDS - 1].argmax(1), bid_te).sum(0)); r['perf|model_last|macro'] = float(mac); r['perf|model_last|weighted'] = float(wei); r['perf|model_last|recall_pc'] = rec
    RES[seed] = r; joblib.dump({'fp': cfg_fp, 'res': r}, pf)
    print(f'  seed {seed} done in {time.time()-t0:.0f}s | benign recall: full {r["perf|full|recall_pc"][0]:.3f}, last-block model {r["perf|model_last|recall_pc"][0]:.3f}, blocked ensemble {r["perf|ens_blocked|recall_pc"][0]:.3f} | cov@0.1: split_last {r["blocked|split_last|0.1|cov"]:.3f}, cvplus {r["blocked|cvplus|0.1|cov"]:.3f}, random cvplus {r["random|cvplus|0.1|cov"]:.3f}', flush=True)
print(f'all seeds done in {(time.time()-t_all)/60:.1f} min')

## 5. Statistics helpers

In [ ]:
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (float('nan') if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); s = d.std(ddof=1) if len(d) > 1 else 0
    return float(d.mean() / s) if s > 0 else float('nan')
def one_sample_p(x, mu):
    x = np.asarray(x, float)
    if len(x) < 2 or x.std(ddof=1) == 0: return float('nan')
    return float(stats.ttest_1samp(x, mu).pvalue)
def fp(p): return '< 0.0001' if (p == p and p < 5e-5) else (f'{p:.4f}' if p == p else 'n/a')
ms = lambda x, d=3: f'{np.mean(x):.{d}f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.{d}f}'
V = lambda key: np.array([RES[s][key] for s in SEEDS]); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); T = {}

## 6. Where the benign problem lives, and what the fix does to coverage

In [ ]:
print('6a. Out-of-fold recall per class and time block (blocked scheme; mean +/- std over seeds). Block 1 = earliest, block %d = latest 1/%d of every series. If benign recall collapses only in the last block, the model needs the recent data.' % (K_FOLDS, K_FOLDS))
rows = []
for k in range(K_FOLDS): rows.append({'held-out block': k + 1, **{c: ms(V('oofrecall|blocked')[:, k, i]) for i, c in enumerate(class_names)}})
rows.append({'held-out block': 'random folds (all)', **{c: ms(V('oofrecall|random').mean(1)[:, i]) for i, c in enumerate(class_names)}})
T['fold_recall'] = pd.DataFrame(rows); print(T['fold_recall'].to_string(index=False))

print('\n6b. Point performance on the test split (burst-aggregated confusion; mean +/- std over seeds)')
rows = []
for nm, lab in [('full', 'full-train model (NB2)'), ('model_last', 'model for the last block (= NB11 recipe)'), ('ens_blocked', 'mean of the %d blocked fold models' % K_FOLDS), ('ens_random', 'mean of the %d random-fold models' % K_FOLDS)]:
    rows.append({'model': lab, 'weighted F1': ms(V(f'perf|{nm}|weighted'), 4), 'macro F1': ms(V(f'perf|{nm}|macro'), 4), **{f'{c} recall': ms(V(f'perf|{nm}|recall_pc')[:, i]) for i, c in enumerate(class_names)}})
T['ensemble'] = pd.DataFrame(rows); print(T['ensemble'].to_string(index=False))
print('   ensemble blocked vs last-block model, weighted F1: p = %s, dz %.2f | benign recall: p = %s' % (fp(ptest(V('perf|ens_blocked|weighted'), V('perf|model_last|weighted'))), dz(V('perf|ens_blocked|weighted'), V('perf|model_last|weighted')), fp(ptest(V('perf|ens_blocked|recall_pc')[:, 0], V('perf|model_last|recall_pc')[:, 0]))))

print('\n6c. Coverage and set size on the temporal test split (mean +/- std over seeds; burst CI = mean of the seeds'' bounds). Benign = coverage of true benign rows.')
CASES = [('blocked', 'split_last'), ('blocked', 'cvplus'), ('blocked', 'cvplus_cc'), ('random', 'cvplus'), ('random', 'cvplus_cc'), ('blocked', 'full_lastq'), ('blocked', 'full_pooledq')]
rows = []; pl = []
for a in ALPHAS:
    for sch, m in CASES:
        k = f'{sch}|{m}|{a:g}|'; cov = V(k + 'cov'); pl.append(one_sample_p(cov, 1 - a))
        rows.append({'alpha': a, 'folds': sch, 'method': m, 'nominal': 1 - a, 'coverage': ms(cov), 'burst CI 95%': f'[{V(k+"cov_lo").mean():.3f}, {V(k+"cov_hi").mean():.3f}]', 'benign coverage': ms(V(k + 'cov_pc')[:, 0]), 'worst-class coverage': ms(V(k + 'worst_cov')),
                     'mean set size': ms(V(k + 'size'), 2), 'singleton share': ms(V(k + 'single')), 'empty share': ms(V(k + 'empty'))})
ph = holm(pl)
for i, rw in enumerate(rows): rw['p vs nominal (Holm)'] = fp(ph[i])
T['coverage'] = pd.DataFrame(rows); print(T['coverage'].to_string(index=False))
print('   Read the burst CI, not the seed p-value (the seed test treats the test split as fixed). random | cvplus is the exchangeable-calibration control: whatever it still misses is temporal shift.')

print('\n6d. Per-class coverage at alpha 0.1 and 0.05 (mean over seeds)')
rows = []
for a in (0.1, 0.05):
    for sch, m in CASES:
        rows.append({'alpha': a, 'folds': sch, 'method': m, **{c: ms(V(f'{sch}|{m}|{a:g}|cov_pc')[:, i]) for i, c in enumerate(class_names)}})
T['perclass'] = pd.DataFrame(rows); print(T['perclass'].to_string(index=False))

print('\n6e. Paired comparison against the NB11 recipe (blocked | split_last), paired over seeds, Holm within this table; dz = paired effect size')
rows = []; pl = []
for a in ALPHAS:
    for sch, m in CASES[1:]:
        for nm, key, idx in [('coverage', 'cov', None), ('benign coverage', 'cov_pc', 0), ('worst-class coverage', 'worst_cov', None), ('set size', 'size', None)]:
            g = lambda s_, m_: (V(f'{s_}|{m_}|{a:g}|{key}')[:, idx] if idx is not None else V(f'{s_}|{m_}|{a:g}|{key}'))
            x, y = g(sch, m), g('blocked', 'split_last'); pl.append(ptest(x, y))
            rows.append({'alpha': a, 'folds': sch, 'method': m, 'metric': nm, 'difference vs split_last': f'{(x - y).mean():+.4f}', 'dz': f'{dz(x, y):.2f}'})
ph = holm(pl)
for i, rw in enumerate(rows): rw['p (Holm)'] = fp(ph[i])
T['compare'] = pd.DataFrame(rows); print(T['compare'].to_string(index=False))

## 7. Figures and saved outputs

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 10)); nom = np.array([1 - a for a in ALPHAS]); sty = {('blocked', 'split_last'): 'k', ('blocked', 'cvplus'): 'C0', ('blocked', 'cvplus_cc'): 'C1', ('random', 'cvplus'): 'C2', ('blocked', 'full_pooledq'): 'C4'}
for (sch, m), c_ in sty.items():
    ax[0, 0].errorbar(nom, [V(f'{sch}|{m}|{a:g}|cov').mean() for a in ALPHAS], yerr=[V(f'{sch}|{m}|{a:g}|cov').std(ddof=1) for a in ALPHAS], marker='o', capsize=3, color=c_, label=f'{sch} {m}')
    ax[0, 1].errorbar(nom, [V(f'{sch}|{m}|{a:g}|cov_pc')[:, 0].mean() for a in ALPHAS], yerr=[V(f'{sch}|{m}|{a:g}|cov_pc')[:, 0].std(ddof=1) for a in ALPHAS], marker='o', capsize=3, color=c_)
    ax[1, 0].errorbar(nom, [V(f'{sch}|{m}|{a:g}|size').mean() for a in ALPHAS], yerr=[V(f'{sch}|{m}|{a:g}|size').std(ddof=1) for a in ALPHAS], marker='o', capsize=3, color=c_)
ax[0, 0].plot([0.98, 1], [0.98, 1], 'k:', lw=1); ax[0, 0].set_title('Test coverage vs nominal'); ax[0, 0].legend(fontsize=7); ax[0, 1].plot([0.98, 1], [0.98, 1], 'k:', lw=1); ax[0, 1].set_title('Coverage of true benign rows')
ax[1, 0].set_title('Mean set size')
for a_ in (ax[0, 0], ax[0, 1], ax[1, 0]): a_.set_xlabel('nominal coverage 1 - alpha'); a_.grid(alpha=.3)
M = np.array([V('oofrecall|blocked')[:, k, :].mean(0) for k in range(K_FOLDS)]); im = ax[1, 1].imshow(M, vmin=0, vmax=1, aspect='auto'); ax[1, 1].set_xticks(range(6)); ax[1, 1].set_xticklabels(class_names, fontsize=8); ax[1, 1].set_yticks(range(K_FOLDS)); ax[1, 1].set_yticklabels([f'block {k+1}' for k in range(K_FOLDS)]); ax[1, 1].set_title('Out-of-fold recall per held-out time block')
for k in range(K_FOLDS):
    for c in range(6): ax[1, 1].text(c, k, f'{M[k, c]:.2f}', ha='center', va='center', fontsize=8, color='w' if M[k, c] < 0.6 else 'k')
plt.tight_layout(); plt.savefig(f'conformalfix_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'ALPHAS': ALPHAS, 'K_FOLDS': K_FOLDS, 'GAP_FRAC': GAP_FRAC, 'N_BOOT': N_BOOT}}, f'conformalfix_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in T.items(): t_.to_csv(f'conformalfix_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB11b', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb11b_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'conformalfix_results_{PIPELINE_VERSION}.pkl', f'conformalfix_{PIPELINE_VERSION}.png', f'nb11b_run_manifest_{PIPELINE_VERSION}.json'] + [f'conformalfix_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)